In [ ]:
!pip install shap transformers librosa datasets

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import shap
import matplotlib.pyplot as plt

from datasets import load_dataset, Audio
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

import os
from scipy.interpolate import interp1d

### Δείγμα αναφοράς

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

model.eval()

### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    # ⚠️ ΑΥΤΟ ΤΟ ΒΗΜΑ είναι πιθανότατα αυτό που λείπει:
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

### Επιλογή δείγματος και πρόβλεψη

In [ ]:
sample_index = TARGET_IDX

sample = test_dataset[sample_index]
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
true_label = sample["emotion"]

In [ ]:
inputs = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
predicted_class = probabilities.argmax(-1).item()

print("True label:", true_label)
print("Predicted label:", model.config.id2label[predicted_class])


In [ ]:
n_segments = 50

### Wrapper για forward pass

In [ ]:
#Wrapper για την επιστροφή raw logits
class ModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_values):
        return self.model(input_values).logits

wrapped_model = ModelWrapper(model)
wrapped_model.eval()

### Δημιουργία background distribution

In [ ]:
input_values = inputs["input_values"]
target_length = input_values.shape[1]

# Τα πρώτα 5 δείγματα του test set χρησιμοποιούνται ως background distribution
# για το DeepExplainer
n_background = 5
background_samples = []

for i in range(n_background):
    a = test_dataset[i]["audio"]["array"]
    inp = feature_extractor(
        a, sampling_rate=16000, return_tensors="pt",
        padding="max_length", max_length=target_length, truncation=True
    )
    background_samples.append(inp["input_values"])

background_tensor = torch.cat(background_samples, dim=0)
print(background_tensor.shape)

In [ ]:
target_input = feature_extractor(
    audio, sampling_rate=sampling_rate, return_tensors="pt",
    padding="max_length", max_length=target_length, truncation=True
)["input_values"]

print(target_input.shape)

### Υπολογισμός DeepSHAP

In [ ]:
explainer_deep = shap.DeepExplainer(wrapped_model, background_tensor)

shap_values_deep = explainer_deep.shap_values(target_input, check_additivity=False)

print(type(shap_values_deep))
print(np.array(shap_values_deep).shape)

In [ ]:
# Ομαδοποίηση attribution ανά sample σε n_segments για το plotting
shap_vals_deep = np.array(shap_values_deep)[0, :, predicted_class]

deep_per_segment = np.array([
    seg.mean() for seg in np.array_split(shap_vals_deep, n_segments)
])

print(deep_per_segment.shape)

In [ ]:
expected_value = explainer_deep.expected_value

if isinstance(expected_value, (list, np.ndarray)):
    base_value = expected_value[predicted_class]
else:
    base_value = expected_value

print("Base value:", base_value)

### Οπτικοποίηση αποτελεσμάτων

In [ ]:
plt.figure(figsize=(14, 4))
plt.bar(np.arange(n_segments), deep_per_segment, color='steelblue')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel("Audio Segment")
plt.ylabel("DeepSHAP Value")
plt.title(f"DeepSHAP explanation for {model.config.id2label[predicted_class]} (True: {true_label})")
plt.tight_layout()
plt.show()

### Αποθήκευση attribution

In [ ]:
# Κανονικοποίηση attribution που είναι ήδη σε μορφή n_segments τιμών
def normalise_attribution(attr):
    attr = np.abs(attr)
    return attr / (attr.sum() + 1e-10)

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)
# Χρησιμοποίησε το ήδη υπολογισμένο deep_per_segment
deepSHAP_attr = normalise_attribution(deep_per_segment)

np.save(f"{save_dir}/DeepSHAP_idx{TARGET_IDX}.npy", deepSHAP_attr)
print(f"Αποθηκεύτηκε: DeepSHAP_idx{TARGET_IDX}.npy, shape={deepSHAP_attr.shape}")

print(os.listdir(save_dir))